In [1]:
from pathlib import Path
DATA_DIR = Path.cwd().parent.parent / 'data' / 'interim' 
DATA_DIR
PATH_AI = DATA_DIR / 'ai_all_followers.csv'
PATH_CHATGPT = DATA_DIR / 'chatgpt_all_followers.csv'
PATH_ML = DATA_DIR / 'ml_all_followers.csv'
PATH_OUTPUT = DATA_DIR.parent / 'processed' /'followers.csv'

In [8]:
import pandas as pd
mandatory_ai = set(pd.read_csv(r"C:\Users\pasqu\Desktop\progettoasnm\Code\data_extraction\data\raw\AI\data.csv")["thread_user_pk"].to_list())
mandatory_chatgpt = set(pd.read_csv(r"C:\Users\pasqu\Desktop\progettoasnm\Code\data_extraction\data\raw\ChatGPT\data.csv")["user_threads_userpk"].to_list())
mandatory_ml = set(pd.read_csv(r"C:\Users\pasqu\Desktop\progettoasnm\Code\data_extraction\data\raw\ML\data.csv")["user_pk"].to_list())

In [11]:
"""
Script per generare il file 'followers.csv' a partire dai tre file:
  - ai_all_followers.csv
  - chatgpt_all_followers.csv
  - ml_all_followers.csv

Obiettivi:
1. Esattamente 51647 archi (righe).
2. Rapporto archi/nodi ≃ 2.5, con nodi = unici di thread_user_pk + thread_follower_pk.
3. Favorire archi che coinvolgono nodi di alto grado su entrambi i lati.
4. Includere **obbligatoriamente** i seguenti thread_user_pk (da ciascun file data):
"""

import pandas as pd

def main():
    INPUT_PATHS = [
        PATH_AI ,
        PATH_CHATGPT ,
        PATH_ML
    ]
    mandatory_nodes = mandatory_ai | mandatory_chatgpt | mandatory_ml
    
    # --- Parametri ---
    TARGET_EDGES = 50000
    RAND         = 42
    
    # -- 1) Carico e concateno --
    dfs = [pd.read_csv(p) for p in INPUT_PATHS]
    df = pd.concat(dfs, ignore_index=True)
    # Rimuovo eventuali colonne "count"
    df = df.drop(columns=[c for c in df.columns if "count" in c.lower()], errors="ignore")

    # Assumo questi nomi di colonna
    U = "thread_user_pk"
    F = "thread_follower_pk"

    # --- 2) Calcolo grado di ogni nodo ---
    deg_u = df[U].value_counts().rename("deg_u")
    deg_f = df[F].value_counts().rename("deg_f")
    df = df.merge(deg_u, left_on=U, right_index=True)
    df = df.merge(deg_f, left_on=F, right_index=True)
    df["weight"] = df["deg_u"] + df["deg_f"]

    # --- 3) Seleziono 1 arco per ogni nodo mandatorio (se presente) ---
    mandatory_idxs = set()
    for node in mandatory_nodes:
        # prendo tutti gli indici in cui appare come thread_user_pk
        idxs = df.index[df[U] == node].tolist()
        if not idxs:
            continue
        # scelgo quello di massimo weight
        best = df.loc[idxs, "weight"].idxmax()
        mandatory_idxs.add(best)

    mand_df = df.loc[sorted(mandatory_idxs)]
    print(f"Inclusi {len(mandatory_idxs)} archi obbligatori (nodi trovati: {len(mandatory_idxs)})")

    # --- 4) Greedy sui restanti per ratio edges/nodes ≤ 2.5 ---
    rest_df = df.drop(index=mandatory_idxs)
    rest_sorted = rest_df.sort_values("weight", ascending=False)

    selected = mand_df.to_dict("records")
    nodes = set(mand_df[U]) | set(mand_df[F])

    for _, row in rest_sorted.iterrows():
        if len(selected) >= TARGET_EDGES:
            break
        u,v = row[U], row[F]
        new_nodes = {u,v} - nodes
        e_new = len(selected) + 1
        n_new = len(nodes) + len(new_nodes)
        selected.append(row)
        nodes |= {u,v}

    # --- 5) Completo casuale se necessario ---
    if len(selected) < TARGET_EDGES:
        need = TARGET_EDGES - len(selected)
        fill = rest_sorted.sample(n=need, random_state=RAND)
        selected.extend(fill.to_dict("records"))

    # --- 6) Salvo e stampo check finale ---
    out_df = pd.DataFrame(selected).sample(frac=1, random_state=RAND)
    total_edges = len(out_df)
    total_nodes = out_df[U].nunique() + out_df[F].nunique()
    print(f"Edges: {total_edges}, Nodes: {total_nodes}, Ratio: {total_edges/total_nodes:.3f}")

    out_df.drop(columns=["deg_u","deg_f","weight"], inplace=True)
    out_df.to_csv(PATH_OUTPUT, index=False)
    print(f"👉 '{PATH_OUTPUT}' creato con successo.")

if __name__ == "__main__":
    main()

Inclusi 81 archi obbligatori (nodi trovati: 81)
Edges: 50000, Nodes: 20564, Ratio: 2.431
👉 'c:\Users\pasqu\Desktop\progettoasnm\Code\data_extraction\data\processed\followers.csv' creato con successo.
